In [1]:
from pathlib import Path
import sys

# common/ lives in BuildingWithClaudeAPI/, a sibling of this folder.
# Walk up from the current directory until BuildingWithClaudeAPI/common is found.
sys.path.append(r"C:\Users\SValathalaKu\Data\claude-bedrock-learning\BuildingWithClaudeAPI")
from common.aws_config import aws_env_setup
#from common.bedrock_config import get_anthropic_client

aws_env_setup()
#client, model = get_anthropic_client()


In [2]:
from anthropic import AnthropicBedrock

def get_anthropic_client():

     client = AnthropicBedrock(
      aws_region="ap-southeast-2"
      )
     model = "au.anthropic.claude-haiku-4-5-20251001-v1:0"
     return client, model
client, model = get_anthropic_client()

In [3]:
from pathlib import Path
from pypdf import PdfReader

def read_pdf_file(file_name: str) -> str:
    """Return the extracted text of a PDF located in the same directory as this notebook."""
    if not file_name.lower().endswith(".pdf"):
        file_name += ".pdf"

    pdf_path = Path.cwd() / file_name
    if not pdf_path.is_file():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    reader = PdfReader(pdf_path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def read_text_file(file_name: str) -> str:
    """Return the contents of a text file in the same directory as this notebook, read as UTF-8."""
    text_path = Path.cwd() / file_name
    if not text_path.is_file():
        raise FileNotFoundError(f"File not found: {text_path}")

    with open(text_path, "r", encoding="utf-8") as f:
        return f.read()

In [11]:
# Define the TOOLS , tell claude that the TOOLS Exists 

tools = [

    {
        "name" : "read_text_file",
        "description" : (
            "Read a UTF-8 text file from the current project directory. "
            "Use when the user asks about the contents of a specific .txt file. "
            "This tool is read-only and cannot modify files."

        ),
        "input_schema": {
            "type": "object",
            "properties" : {
                "file_name" : {
                    "type": "string",
                    "description" : "Text file name e.g xxx.txt"
                }
            },
            "required" : ["file_name"],
            "additionalProperties": False 
        }

    },
    {
        "name": "read_pdf_file",
        "description": (
            "Read and extract the text of a PDF file from the current project directory. "
            "Use when the user asks about the contents of a specific .pdf file. "
            "This tool is read-only and cannot modify files."
        ),
        "strict":True,
        "input_schema": {
            "type": "object",
            "properties": {
                "file_name": {
                    "type": "string",
                    "description": "PDF file name, e.g. report.pdf. The .pdf extension is optional."
                }
            },
            "required": ["file_name"],
            "additionalProperties": False
        }
    }

]


In [12]:
messages = [
    {
        "role": "user",
        "content": "Read the pdf file sajin.txt in the directory and sumarise in 3 main points "
    }
]



# First call to claude 

response = client.messages.create(
    model = model, 
    max_tokens= 300,
    tools = tools,
    tool_choice= {"type": "auto"},
    messages=messages
)


In [6]:
print(response)

Message(id='msg_bdrk_6kvzvu2vpokfh3wiq2tcx6p6p5zfyfkzsjkoxykgtopwqqlsec6a', container=None, content=[TextBlock(citations=None, text='I\'ll read the file for you. Note that you mentioned "sajin.txt" which is a text file, so I\'ll read it as a text file rather than a PDF.', type='text'), ToolUseBlock(id='toolu_bdrk_01LmDzcvjjZ7MmkWfooPT4Vu', caller=None, input={'filename': 'sajin.txt'}, name='read_text_file', type='tool_use', toolset_name=None)], diagnostics=None, model='anthropic.claude-haiku-4-5-20251001-v1:0', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=759, output_tokens=97, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [13]:
# When Claude calls a tool, the response has stop_reason "tool_use"
# and the content array contains a tool_use block alongside any text.
print(f"first stop_reason: {response.stop_reason}")

# Find the tool_use block. A response might contain text blocks before the
# tool_use block, so scan the content array rather than assuming position.
tool_use = next(block for block in response.content if block.type == "tool_use")
print(f"Tool: {tool_use.name}")
print(f"Input: {tool_use.input}")

first stop_reason: tool_use
Tool: read_text_file
Input: {'file_name': 'sajin.txt'}


In [14]:
# 4. Preserve Claude's tool request.
messages.append({
    "role": "assistant",
    "content": response.content
})

In [15]:
# Application executes the tool 
# Map Claude tool names to Python functions
import json 
tool_functions = {
    "read_text_file": read_text_file,
    "read_pdf_file": read_pdf_file
}

tool_results = []

for content in response.content:
    if content.type != "tool_use":
        continue 

    try:
        function = tool_functions.get(content.name)

        if function is None:
            raise ValueError(f"Unknown tool: {content.name}")
        # block.input is a dictionary of arguments
        output = function(**content.input)

        tool_results.append({
            "type": "tool_result",
            "tool_use_id": content.id,
            "content": json.dumps(output)

        })
    except Exception as error:
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": content.id,
            "content": json.dumps({
                "errorCategory": "validation",
                "isRetryable": False,
                "description": str(error)
            }),
            "is_error": True
        })
        
print (tool_results)

[{'type': 'tool_result', 'tool_use_id': 'toolu_bdrk_01MuBdxHLAsDBRrCfLPZBY9d', 'content': '"Strict tool use\\nEnforce JSON Schema compliance on Claude\\u2019s tool inputs with grammar-constrained sampling.\\n\\nSetting strict: true on a tool definition guarantees Claude\\u2019s tool inputs match your JSON Schema by constraining the model\\u2019s token sampling to schema-valid outputs (a technique called grammar-constrained sampling). This page covers why strict mode matters for agents, how to enable it, and common use cases. For the supported JSON Schema subset, see JSON Schema limitations. For non-strict schema guidance, see Define tools.\\n\\nStrict tool use validates tool parameters, ensuring Claude calls your functions with correctly typed arguments. Use strict tool use when you need to:\\n\\nValidate tool parameters\\nBuild agentic workflows\\nEnsure type-safe function calls\\nHandle complex tools with nested properties\\nWhy strict tool use matters for agents\\nBuilding reliable 

In [16]:
# 6. Return the tool result to Claude.
messages.append({
    "role": "user",
    "content": tool_results
})

# 7. Claude uses the file contents to answer.
followup = client.messages.create(
    model=model,
    max_tokens=300,
    tools=tools,
    messages=messages
)

print("Final stop reason:", followup.stop_reason)

for block in followup.content:
    if block.type == "text":
        print(block.text)

Final stop reason: end_turn
Perfect! I've read the file. Here are the **3 main points** from the document:

## Summary of sajin.txt

1. **Strict Tool Use for Validation**: Strict tool use enforces JSON Schema compliance by using grammar-constrained sampling to guarantee that Claude's tool inputs always match the specified schema. This eliminates the need for validation and retry logic, ensuring type-safe function calls and production-ready agents that work consistently at scale.

2. **How It Works Simply**: The implementation involves three steps: (1) defining a JSON schema for the tool's input, (2) setting "strict": true in the tool definition, and (3) handling the guaranteed validated tool calls where inputs strictly follow the schema and tool names are always valid.

3. **Reliability and Data Safety**: Strict tool use is crucial for building reliable agentic systems because it prevents Claude from returning incompatible types (like returning "2" as a string instead of integer) or om